# 10 • Tool Calling: ตรวจสถานการณ์คลังยาง

เป้าหมาย: ดูชื่อ tool และ arguments ที่ LLM เลือก รัน Python tool แล้วส่งผลกลับไปให้ LLM ตอบภาษาไทย

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

บริบทงานอ้างอิงจาก [เว็บไซต์กลุ่มศรีตรัง](https://www.sritranggroup.com/th/home) ส่วนรูปแบบ notebook อ้างอิงจาก [ตัวอย่าง GitHub](https://github.com/biodatlab/cpf-genai-workshop/tree/main/tutorial_notebooks) และใช้ [OpenAI API documentation](https://developers.openai.com/api/docs) สำหรับ syntax

## เตรียมพร้อมก่อนเริ่ม
เพิ่ม Secret ชื่อ `OPENAI_API_KEY` ใน Colab ด้วยตนเอง และเปิด **Notebook access** สำหรับ notebook นี้ โค้ดจะอ่านผ่าน `userdata.get()` โดยไม่แสดงค่า key

รันเซลล์จากบนลงล่าง หลังติดตั้ง package และเตรียม key แล้ว ทดสอบตัวอย่างหลักใน Colab ด้วย API แล้ว แต่ผล LLM อาจเปลี่ยนได้ การผ่านตัวอย่างนี้ไม่ใช่การรับรองทุกคำถามหรือการใช้งานจริง

ใช้ `gpt-4.1-mini` สำหรับสร้างคำตอบ (ไม่เกิน 500 tokens ต่อ request) และ `text-embedding-3-small` สำหรับ Embedding โดย Agent Loop จำกัดไว้ 5 รอบ

In [1]:
%pip -q install openai

In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

In [3]:
def get_flood_level(factory):
    readings = {"คลังยางแท่ง A": 55, "คลังน้ำยาง B": 10}
    if factory not in readings:
        return {"error": "ไม่พบหน่วยงานนี้", "source": "ข้อมูลสมมติสำหรับการอบรม"}
    return {"factory": factory, "water_cm": readings[factory], "source": "ข้อมูลสมมติสำหรับการอบรม", "live": False}

In [4]:
tools = [{
    "type": "function", "name": "get_flood_level",
    "description": "อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B",
    "parameters": {"type": "object", "properties": {
        "factory": {"type": "string", "enum": ["คลังยางแท่ง A", "คลังน้ำยาง B"]}
    }, "required": ["factory"], "additionalProperties": False},
    "strict": True,
}]

In [5]:
import json

messages = [{"role": "user", "content": "ระดับน้ำสมมติที่คลังยางแท่ง A เป็นเท่าไร?"}]
response = client.responses.create(
    model=MODEL, input=messages, tools=tools,
    tool_choice={"type": "function", "name": "get_flood_level"},
    max_output_tokens=500,
)
messages += response.output
for call in response.output:
    if call.type == "function_call":
        print("Tool ที่เลือก:", call.name, "Arguments:", call.arguments)
        result = get_flood_level(**json.loads(call.arguments))
        print("ผลลัพธ์:", result)
        messages.append({"type": "function_call_output", "call_id": call.call_id,
                         "output": json.dumps(result, ensure_ascii=False)})

Tool ที่เลือก: get_flood_level Arguments: {"factory":"คลังยางแท่ง A"}
ผลลัพธ์: {'factory': 'คลังยางแท่ง A', 'water_cm': 55, 'source': 'ข้อมูลสมมติสำหรับการอบรม', 'live': False}


In [6]:
final = client.responses.create(
    model=MODEL, input=messages,
    instructions="ตอบเป็นภาษาไทยและระบุชัดเจนว่าค่าที่อ่านเป็นข้อมูลสมมติสำหรับการอบรม",
    max_output_tokens=500,
)
print(final.output_text)

ระดับน้ำสมมติที่คลังยางแท่ง A คือ 55 เซนติเมตร โดยข้อมูลนี้เป็นข้อมูลสมมติสำหรับการอบรมค่ะ


## ลองทำด้วยตนเอง
1. ถามเกี่ยวกับคลังน้ำยาง B และดู arguments
2. เปลี่ยนเป็น tool_choice='auto' แล้วลองทักทายแทนคำถามสถานะ
3. อธิบายว่าส่วนใดเลือก tool และส่วนใดรัน Python

ค่าระดับน้ำเป็น sample data ไม่ใช่สถานการณ์จริง